In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
#from lazypredict.Supervised import LazyClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("term-deposit-marketing-2020.csv")

df["y"] = df["y"].map({"no": 0, "yes": 1})

binary_cols = ["housing", "loan", "default"]  # example

for col in binary_cols:
    df[col] = df[col].map({"no": 0, "yes": 1})

#df = df.drop(columns="contact")

df.replace('unknown', np.nan, inplace=True)


#binary_cols = ["housing", "loan", "default"]  # example

#for col in binary_cols:
#    df[col] = df[col].map({"no": 0, "yes": 1})

df_sub = df[df["y"] == 1].copy()

In [2]:
cluster_features = [
    "age",
    "job",
    #"education",
    "marital",
    "default",
    "balance",
    "housing",
    "loan"
]

In [3]:
X_cluster = df_sub[cluster_features].copy()

In [4]:
#X_cluster.isna().sum()

In [5]:
X_cluster.info()

<class 'pandas.DataFrame'>
Index: 2896 entries, 83 to 39997
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   age      2896 non-null   int64
 1   job      2879 non-null   str  
 2   marital  2896 non-null   str  
 3   default  2896 non-null   int64
 4   balance  2896 non-null   int64
 5   housing  2896 non-null   int64
 6   loan     2896 non-null   int64
dtypes: int64(5), str(2)
memory usage: 181.0 KB


In [6]:
#raise Exception('End of code for now.')

In [7]:
X_cluster = pd.get_dummies(
    X_cluster,
    columns=["job", "marital"],
    dtype=int
)

In [8]:
#df_sub.info()

In [9]:
#X_cluster.isna().sum()

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

In [11]:
kmeans = KMeans(
    n_clusters=5,
    random_state=1234,
    n_init="auto"
)

clusters = kmeans.fit_predict(X_scaled)

In [12]:
df_sub["cluster"] = clusters
X_cluster["cluster"] = df_sub["cluster"]

In [13]:
X_cluster.groupby("cluster")[["age", "balance", "housing", "loan", 'default']].mean()

,age,balance,housing,loan,default
cluster,,,,,
0,62.615894,2317.695364,0.178808,0.092715,0.000000
1,27.073171,1657.939024,0.231707,0.000000,0.000000
2,41.968118,1400.231726,0.561431,0.153966,0.019440
3,33.129237,1644.792373,0.533898,0.128178,0.021186
4,42.658199,1757.457275,0.443418,0.108545,0.009238


In [14]:
#df_sub.groupby("cluster").mean(numeric_only=True)
df_sub.groupby('cluster')[['day', 'duration', 'campaign']].mean()

,day,duration,campaign
cluster,,,
0,16.907285,566.026490,2.198675
1,17.804878,464.195122,2.329268
2,15.521773,721.930016,2.507776
3,16.021186,680.206568,2.290254
4,15.558891,655.420323,2.498845


In [15]:
#X_cluster.groupby(df_sub["cluster"]).mean() #if cluster only defined within df_sub

In [16]:
X_cluster.groupby("cluster")[['job_admin','job_blue-collar','job_entrepreneur','job_housemaid','job_management',
                              'job_retired','job_self-employed','job_services','job_student','job_technician',
                              'job_unemployed']].mean()

,job_admin,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed
cluster,,,,,,,,,,,
0,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.0,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.0,0.000000,0.000000
2,0.168740,0.300156,0.053655,0.032659,0.000000,0.0,0.050544,0.119751,0.0,0.217729,0.047434
3,0.141949,0.157839,0.020127,0.011653,0.257415,0.0,0.049788,0.088983,0.0,0.229873,0.037076
4,0.000000,0.000000,0.000000,0.000000,1.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000


In [17]:
df_sub.groupby("cluster")["job"].value_counts(normalize=True)

cluster  job          
0        retired          1.000000
1        student          1.000000
2        blue-collar      0.302983
         technician       0.219780
         admin            0.170330
         services         0.120879
         entrepreneur     0.054160
         self-employed    0.051020
         unemployed       0.047881
         housemaid        0.032967
3        management       0.258786
         technician       0.231097
         blue-collar      0.158679
         admin            0.142705
         services         0.089457
         self-employed    0.050053
         unemployed       0.037274
         entrepreneur     0.020234
         housemaid        0.011715
4        management       1.000000
Name: proportion, dtype: float64

In [18]:
X_cluster.groupby("cluster")[['marital_single','marital_married','marital_divorced']].mean()

,marital_single,marital_married,marital_divorced
cluster,,,
0,0.026490,0.662252,0.311258
1,0.951220,0.048780,0.000000
2,0.000000,0.804821,0.195179
3,1.000000,0.000000,0.000000
4,0.002309,0.782910,0.214781


In [19]:
df_sub.groupby("cluster")["marital"].value_counts(normalize=True)

cluster  marital 
0        married     0.662252
         divorced    0.311258
         single      0.026490
1        single      0.951220
         married     0.048780
2        married     0.804821
         divorced    0.195179
3        single      1.000000
4        married     0.782910
         divorced    0.214781
         single      0.002309
Name: proportion, dtype: float64